# Task 2 — Resolution scaling and crop-then-click

**Owner:** Dong Yiwei  
**Hypothesis (H2):** Higher effective resolution and a two-stage crop-then-click policy will improve ScreenSpot-Pro most, especially on small targets — recovering 5-10 pp for generalists.

**Variations (5 cells)**
1. Single-stage at 0.5× native
2. Single-stage at 1× native (proposal default)
3. Single-stage at 2× native
4. Crop-then-click, 512×512 window
5. Crop-then-click, 768×768 window

**Deeper analysis:** failure analysis on small icons + ablation of resolution-only vs. crop-only gains.


In [ ]:
# Colab bootstrap. On local Jupyter this is a no-op because the env is already set up.
import sys
from pathlib import Path

REPO_URL = "https://github.com/ali-epita/action-learning-ais5.git"
REPO_DIR = "/content/action-learning-ais5"

if "google.colab" in sys.modules:
    if not Path(REPO_DIR).exists():
        !git clone --depth 1 {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

    !pip install -q -e . --no-deps
    # Strip Colab's pre-installed torchao (peft 0.17+ raises ImportError on the
    # 0.10.0 that ships with Colab instead of treating it as absent). LoRA
    # fine-tuning here does not depend on torchao.
    !pip uninstall -y -q torchao
    !pip install -q "transformers>=4.49,<5" "accelerate>=0.34" "datasets>=3.0" \
                    "peft>=0.13" "qwen-vl-utils>=0.0.10" \
                    rich tqdm pyyaml typer pillow huggingface_hub safetensors psutil

    src_dir = str(Path(REPO_DIR) / "src")
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    import importlib
    importlib.invalidate_caches()

import ais5
print(f"ais5 v{ais5.__version__} ready  (colab={'google.colab' in sys.modules})")


In [ ]:
import copy
import json
import time
from pathlib import Path

import pandas as pd

from ais5.data import load_benchmark
from ais5.eval import evaluate_model
from ais5.eval.breakdown import by_target_size
from ais5.eval.click import ClickResult, point_in_bbox
from ais5.models import get_model
from ais5.tile import CropConfig, crop_then_click, scale_image
from ais5.utils import set_global_seed, setup_logging

setup_logging()
set_global_seed(42)


In [ ]:
# Toggle. SMOKE_MODE=True exercises one factor + one crop on 25 samples so the
# pipeline runs end-to-end in a few minutes. Flip to False for the H2 sweep.
SMOKE_MODE = True

if SMOKE_MODE:
    FACTORS     = [1.0]
    CROP_SIZES  = [512]
    LIMIT       = 25
else:
    FACTORS     = [0.5, 1.0, 2.0]
    CROP_SIZES  = [512, 768]
    LIMIT       = None

BASE_MODEL    = 'qwen2.5-vl-3b'
ADAPTER_PATH  = None   # set to e.g. 'checkpoints/qwen-r32-50000' to plug in Task 1's best
BENCHMARK     = 'screenspot-pro'

results_dir = Path('results/tile')
results_dir.mkdir(parents=True, exist_ok=True)
jsonl_path = results_dir / 'sweep.jsonl'
jsonl_path.unlink(missing_ok=True)

print(f"SMOKE_MODE={SMOKE_MODE}  factors={FACTORS}  crops={CROP_SIZES}  limit={LIMIT}")

model = get_model(BASE_MODEL, peft_adapter=ADAPTER_PATH)
samples_pro = list(load_benchmark(BENCHMARK))
print(f"loaded {len(samples_pro)} {BENCHMARK} samples")


## Resolution sweep


In [ ]:
def _as_jsonable_result(r):
    return {
        'sample_id': r.sample_id,
        'pred': list(r.pred) if r.pred is not None else None,
        'bbox': list(r.bbox),
        'correct': r.correct,
        'benchmark': r.benchmark,
        'target_type': r.target_type,
        'ui_type': r.ui_type,
        'target_relative_area': r.target_relative_area,
        'raw_response': r.raw_response,
        'latency_ms': r.latency_ms,
    }


# Resolution sweep: scale the screenshot before inference, rescale the bbox to
# match so accuracy is measured in the same coordinate system the model sees.
rows = []
resolution_results = {}
selected_samples = samples_pro if LIMIT is None else samples_pro[:LIMIT]

for factor in FACTORS:
    samples = []
    for s in selected_samples:
        scaled = copy.copy(s)
        scaled.image = scale_image(s.image, factor)
        scaled.image_size = scaled.image.size
        x1, y1, x2, y2 = s.bbox
        scaled.bbox = (x1 * factor, y1 * factor, x2 * factor, y2 * factor)
        samples.append(scaled)
    run = evaluate_model(model, samples, benchmark=BENCHMARK, limit=None)
    resolution_results[factor] = run.results
    per_sample_path = results_dir / f'resolution_{factor:g}x_per_sample.jsonl'
    with per_sample_path.open('w') as f:
        for r in run.results:
            f.write(json.dumps({**_as_jsonable_result(r), 'factor': factor}) + '\n')
    row = {
        'strategy': 'resolution',
        'factor': factor,
        'crop': None,
        'n': len(run.results),
        'accuracy': run.accuracy,
        'mean_latency_ms': run.avg_latency_ms,
        'per_sample_path': str(per_sample_path),
    }
    rows.append(row)
    with jsonl_path.open('a') as f:
        f.write(json.dumps(row) + '\n')
    print(
        f"factor={factor}x  acc={run.accuracy:.4f}  (n={len(run.results)})  "
        f"mean_lat={run.avg_latency_ms:.0f} ms  saved={per_sample_path}"
    )


## Crop-then-click sweep

Wraps the model's `predict()` in `crop_then_click(...)` to produce a refined click. We measure end-to-end accuracy *and* per-step latency (since refinement doubles the inference count).


In [ ]:
# Crop-then-click wraps the predict() call in a two-stage zoom-in policy.
# Track end-to-end latency since refinement doubles the inference count.
def _crop_eval(samples, crop_size, limit):
    cfg = CropConfig(crop_size=crop_size)
    selected = samples if limit is None else samples[:limit]
    rs = []
    meta_rows = []
    for s in selected:
        t0 = time.perf_counter()
        out = crop_then_click(model, s.image, s.instruction, cfg=cfg)
        dt_ms = (time.perf_counter() - t0) * 1000.0
        pred = out.parsed.point
        correct = bool(pred is not None and point_in_bbox(pred, s.bbox))
        result = ClickResult(
            sample_id=s.sample_id or '',
            pred=pred,
            bbox=s.bbox,
            correct=correct,
            benchmark=s.benchmark,
            target_relative_area=s.target_relative_area,
            ui_type=s.ui_type,
            target_type=s.target_type,
            raw_response=out.text,
            latency_ms=dt_ms,
        )
        rs.append(result)
        meta_rows.append({
            **_as_jsonable_result(result),
            'crop_size': crop_size,
            'crop_box': out.metadata.get('crop_box'),
            'coarse_point': out.metadata.get('coarse_point'),
            'refined_point': out.metadata.get('refined_point'),
            'refined_coord_frame': out.metadata.get('refined_coord_frame'),
            'parser': out.parsed.parser,
        })
    acc = sum(r.correct for r in rs) / len(rs) if rs else 0.0
    return rs, meta_rows, acc


crop_results = {}
crop_meta = {}
for crop in CROP_SIZES:
    rs, meta_rows, acc = _crop_eval(samples_pro, crop_size=crop, limit=LIMIT)
    crop_results[crop] = rs
    crop_meta[crop] = meta_rows
    per_sample_path = results_dir / f'crop_{crop}_per_sample.jsonl'
    with per_sample_path.open('w') as f:
        for item in meta_rows:
            f.write(json.dumps(item) + '\n')
    row = {
        'strategy': 'crop_then_click',
        'factor': 1.0,
        'crop': crop,
        'n': len(rs),
        'accuracy': acc,
        'mean_latency_ms': sum(r.latency_ms for r in rs) / max(1, len(rs)),
        'per_sample_path': str(per_sample_path),
    }
    rows.append(row)
    with jsonl_path.open('a') as f:
        f.write(json.dumps(row) + '\n')
    print(
        f"crop={crop}px  acc={acc:.4f}  (n={len(rs)})  "
        f"mean_lat={row['mean_latency_ms']:.0f} ms  saved={per_sample_path}"
    )


## Combined sweep table


In [ ]:
df = pd.DataFrame(rows)
df.to_csv(results_dir / 'sweep.csv', index=False)
df


## Failure analysis on small icons

Group results by target-area bucket. The hypothesis is that the `xs` and `s` buckets benefit most from crop-then-click.


In [ ]:
# Failure analysis on small icons. H2 says the xs/s buckets benefit most from
# crop-then-click. Save each crop variant's bucket-level breakdown.
size_breakdowns = []
for crop, rs in crop_results.items():
    print(f"\n=== crop_size={crop}px ===")
    bd = by_target_size(rs)
    bd.insert(0, 'crop', crop)
    size_breakdowns.append(bd)
    display(bd)

if size_breakdowns:
    size_df = pd.concat(size_breakdowns, ignore_index=True)
    size_df.to_csv(results_dir / 'target_size_breakdown.csv', index=False)
    display(size_df)
